In [17]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else
    "cpu"
)
print(f"Using device: {device}")


Using device: mps


In [18]:
model_path = "../models"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
print("Model loaded successfully")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded successfully


In [19]:
def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


In [20]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(targets[0], skip_special_tokens=True)


In [21]:
test_dialogue = """
Amanda: I baked cookies. Do you want some?
Jerry: Sure! What kind did you make?
Amanda: Chocolate chip and oatmeal raisin.
Jerry: Chocolate chip please! Are you at home now?
Amanda: Yes, but I will be leaving for campus in 20 minutes. I can drop them off at your dorm.
Jerry: That would be awesome, thanks Amanda!
"""

print(summarize_dialogue(test_dialogue))

amanda baked cookies and oatmeal raisin. jerry will drop them off at amanda's dorm.
